This notebook creates the spatially clustered CV splits for the aggregated radon survey data + feature space

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# ---------------------------------------------------------------------
# Add project root to Python path
# ---------------------------------------------------------------------

# project root = two levels above notebooks
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
# Load helper functions
from src.data.spatial_clustering import create_spatial_clusters
from src.data.spatial_splitting import create_spatial_test_and_cv_splits


In [ ]:
# Load path information
from config.paths import MAIN_DATASET
from config.paths import SPATIAL_CV_DATASET
from config.paths import SPATIAL_CV_DIR

# make sure those directories exist...
SPATIAL_CV_DIR.mkdir(parents=True, exist_ok=True)
# and that data is there...
if not MAIN_DATASET.exists():
    raise FileNotFoundError(
        f"Main dataset not found: {MAIN_DATASET}"
    )

# Ensure that we aren't overwriting old splits all the time!!!
if SPATIAL_CV_DATASET.exists():
    raise RuntimeError(
        f"Spatial CV split file already exists:\n{SPATIAL_CV_DATASET}\n"
        "Delete it manually if regeneration is intended."
    )

In [ ]:
from config.paths import (
    # core dataset
    MAIN_DATASET,

    # raw inputs
    RADON_CLEANED,
    FSA_BOUNDARY_SHAPEFILE,

    # intermediate features
    FSA_CENTROIDS,
    CENSUS_FSA,
    GEOLOGY_FSA,
    SURFICIAL_FSA,
    URANIUM_FSA,
    HEATING_DAYS,
)

In [ ]:
# --------------------------------------------------
# Spatial CV configuration
# --------------------------------------------------

from config.cv_params import (
    N_SPATIAL_CLUSTERS,  # number of k-means clusters that we use to spatially divide Canada
    NUM_TEST_SPLITS,  # KFOLD SPLITTING NEEDS AN INTEGER: 1/N gives the fractional size of the test split
    NUM_CV_SPLITS, # Number of folds for kfold splitting
    OUTER_FOLD_COLUMN, # name of the column for the cv_fold id
)
from config.feature_groups import (
    RANDOM_STATE # random seed
)


### Split up data between: 
  - Test data (label = '-1')
  - Cross validation data (label = 0 -- N-1, where N is number of folds )
  
  To use for CV, train on data with label (!=0 and != N_{current_CV_round})

In [ ]:
df = pd.read_csv(MAIN_DATASET)

gdf_clusters = create_spatial_clusters(df, n_clusters=N_SPATIAL_CLUSTERS)

df_splits = create_spatial_test_and_cv_splits(
    gdf_clusters,
    n_test_splits=NUM_TEST_SPLITS,
    n_cv_splits=NUM_CV_SPLITS,
    random_state=RANDOM_STATE,
)

# Get rid of ID column from spatial-kfold work
df_splits = df_splits.drop(columns=["spkf_id"])

### Write out data to CSV

In [ ]:
print("Saving dataset to:")
print(SPATIAL_CV_DATASET)
df_splits.to_csv(SPATIAL_CV_DATASET, index=False)

In [ ]:
cluster_summary = (
    df_splits
    .groupby("spatial_cluster")
    .agg(
        n_observations=("spatial_cluster", "size"),
        province=("provinceterritory", "first"),
        is_test=("is_test", "first"),
        cv_fold=("cv_fold", "first")
    )
    .reset_index()
)

In [ ]:
df_splits.head()

In [ ]:
# N_observations per fold
df_splits.groupby("cv_fold").size()

In [ ]:
#Largest cluster:
cluster_summary.sort_values("n_observations", ascending=False).head()

In [ ]:
# clusters per fold
cluster_summary.groupby("cv_fold").size()